In [6]:
# check installed version
import pycaret
pycaret.__version__

'3.3.2'

In [7]:
from data_master_eng_ml.utils.helpers import read_data_from_mongodb

In [8]:
from data_master_eng_ml.config import GAMES_SILVER_COLLECTION,MONGODB_DEFAULT_DATABASE,MLFLOW_TRACKING_URI

In [9]:
df_2022 = read_data_from_mongodb(MONGODB_DEFAULT_DATABASE,GAMES_SILVER_COLLECTION)

In [10]:
df_2022

,id,name,dat_ref,genres_first,has_remaster,target,age_rating_group,games_developed,has_parents,games_published,...,unknown_game_mode,auditory,bird_view_slash_isometric,first_person,side_view,text,third_person,unknown_player_perspectives,virtual_reality,has_global_launch
0,71,Portal,2022,shooter,False,1,15+,126.0,0.0,1397.0,...,0,0,0,1,0,0,0,0,0,1
1,72,Portal 2,2022,adventure,False,1,12+,87.0,0.0,95.0,...,0,0,0,1,0,0,0,0,0,1
2,4250,Kingdom: The Far Reaches,2022,adventure,False,0,12+,5.0,0.0,43.0,...,0,0,0,0,0,0,1,0,0,1
3,6614,Zero Escape: Virtue's Last Reward,2022,point-and-click,False,1,18+,1.0,0.0,132.0,...,0,0,0,1,0,1,0,0,0,1
4,7046,Factorio,2022,strategy,False,1,12+,2.0,0.0,2.0,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4639,308471,Supersonic Mario,2022,sport,False,0,No Rating,NaN,NaN,NaN,...,0,0,0,0,0,0,1,0,0,0
4640,321236,Symphony Of Motion,2022,indie,False,0,No Rating,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,0
4641,307811,Deltatraveler: Section 2,2022,adventure,False,0,No Rating,6.0,0.0,6.0,...,0,0,1,0,0,0,0,0,0,1
4642,286513,Outriders: Complete Edition,2022,unknown_genres_name,False,0,18+,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0


In [11]:
df_2022['continent_name'] = df_2022['continent_name'].str.replace(' ', '_')
df_2022['age_rating_group'] = df_2022['age_rating_group'].str.replace(' ', '_')

In [12]:
df_2022

,id,name,dat_ref,genres_first,has_remaster,target,age_rating_group,games_developed,has_parents,games_published,...,unknown_game_mode,auditory,bird_view_slash_isometric,first_person,side_view,text,third_person,unknown_player_perspectives,virtual_reality,has_global_launch
0,71,Portal,2022,shooter,False,1,15+,126.0,0.0,1397.0,...,0,0,0,1,0,0,0,0,0,1
1,72,Portal 2,2022,adventure,False,1,12+,87.0,0.0,95.0,...,0,0,0,1,0,0,0,0,0,1
2,4250,Kingdom: The Far Reaches,2022,adventure,False,0,12+,5.0,0.0,43.0,...,0,0,0,0,0,0,1,0,0,1
3,6614,Zero Escape: Virtue's Last Reward,2022,point-and-click,False,1,18+,1.0,0.0,132.0,...,0,0,0,1,0,1,0,0,0,1
4,7046,Factorio,2022,strategy,False,1,12+,2.0,0.0,2.0,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4639,308471,Supersonic Mario,2022,sport,False,0,No_Rating,NaN,NaN,NaN,...,0,0,0,0,0,0,1,0,0,0
4640,321236,Symphony Of Motion,2022,indie,False,0,No_Rating,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,0
4641,307811,Deltatraveler: Section 2,2022,adventure,False,0,No_Rating,6.0,0.0,6.0,...,0,0,1,0,0,0,0,0,0,1
4642,286513,Outriders: Complete Edition,2022,unknown_genres_name,False,0,18+,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0


In [13]:
# import ClassificationExperiment and init the class
from pycaret.classification import ClassificationExperiment
balenced_exp = ClassificationExperiment()

In [14]:
import mlflow 
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [15]:
MLFLOW_TRACKING_URI

'http://localhost:5000'

In [16]:
# init setup on exp
balenced_exp.setup(
    df_2022,
    target="target",
    session_id=123,
    log_experiment="mlflow",
    experiment_name="balenced_experiment",
    fold=5,
    log_data=True,
    imputation_type="simple",
    log_plots=True,
    ignore_features=["id", "name","dat_ref"],
    categorical_features=["continent_name","age_rating_group","genres_first"],
    feature_selection=True,
)

[LightGBM] [Info] Number of positive: 264, number of negative: 2986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001958 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 303
[LightGBM] [Info] Number of data points in the train set: 3250, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081231 -> initscore=-2.425741
[LightGBM] [Info] Start training from score -2.425741


,Description,Value
0,Session id,123
1,Target,target
2,Target type,Binary
3,Original data shape,"(4644, 40)"
4,Transformed data shape,"(4644, 8)"
5,Transformed train set shape,"(3250, 8)"
6,Transformed test set shape,"(1394, 8)"
7,Ignore features,3
8,Numeric features,30
9,Categorical features,3


In [17]:
# compare baseline models
best = balenced_exp.compare_models(include=['xgboost','rf','lr'],sort='AUC')

2025/04/12 14:20:30 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/04/12 14:20:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run Extreme Gradient Boosting at: http://localhost:5000/#/experiments/387310387560721782/runs/f7a788f2adb74b3683cd615d48f294a3.
2025/04/12 14:20:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/387310387560721782.
2025/04/12 14:20:31 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/04/12 14:20:31 INFO mlflow.tracking._tracking_service.client: 🏃 View run Random Forest Classifier at: http://localhost:5000/#/experiments/387310387560721782/runs/8bee4b3920c74886a6f8de8f1045c858.
2025/04/12 14:20:31 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#

In [18]:
balenced_exp.finalize_model(best)

2025/04/12 14:20:32 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/04/12 14:20:32 INFO mlflow.tracking._tracking_service.client: 🏃 View run Extreme Gradient Boosting at: http://localhost:5000/#/experiments/387310387560721782/runs/df71043d1c914f148d1059c28372c91a.
2025/04/12 14:20:32 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/387310387560721782.


Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['games_developed', 'has_parents',
                                             'games_published', 'onlinecoopmax',
                                             'onlinemax', 'classic_console',
                                             'less_common_portable_console',
                                             'mobile', 'modern_console',
                                             'others', 'pc', 'portable_console',
                                             'unknown_platforms_name', 'vr',
                                             'battle_royale', 'co_operati...
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None,
                               objective='binary:logistic', ...))],
         verbose=False)

In [ ]:
balenced_exp.

In [19]:
balenced_exp.USI

'fbd0'

In [20]:
experiment_id = mlflow.search_experiments(filter_string=f"name = 'balenced_experiment'")[0].experiment_id

In [21]:
df_runs = mlflow.search_runs(experiment_ids=[experiment_id])

In [22]:
df_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.TT,metrics.MCC,metrics.AUC,metrics.Accuracy,...,tags.Source,tags.Run ID,tags.USI,tags.Run Time,tags.mlflow.runName,tags.mlflow.parentRunId,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.user,tags.URI
0,df71043d1c914f148d1059c28372c91a,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/df71043d1...,2025-04-12 17:20:31.790000+00:00,2025-04-12 17:20:32.854000+00:00,0.190,NaN,NaN,NaN,...,finalize_model,df71043d1c914f148d1059c28372c91a,fbd0,0.29,Extreme Gradient Boosting,39c59009a6b5481fa6399133036d4cb3,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,5cf458be
1,afdda57ceca84a29ac3c48c0af17714c,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/afdda57ce...,2025-04-12 17:20:31.119000+00:00,2025-04-12 17:20:31.308000+00:00,26.238,0.2154,0.7294,0.9212,...,compare_models,afdda57ceca84a29ac3c48c0af17714c,fbd0,131.32,Logistic Regression,39c59009a6b5481fa6399133036d4cb3,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,573ce798
2,8bee4b3920c74886a6f8de8f1045c858,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/8bee4b392...,2025-04-12 17:20:30.829000+00:00,2025-04-12 17:20:31.024000+00:00,13.554,0.4702,0.8180,0.9326,...,compare_models,8bee4b3920c74886a6f8de8f1045c858,fbd0,67.89,Random Forest Classifier,39c59009a6b5481fa6399133036d4cb3,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,68a0ac74
3,f7a788f2adb74b3683cd615d48f294a3,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/f7a788f2a...,2025-04-12 17:20:28.356000+00:00,2025-04-12 17:20:30.727000+00:00,11.288,0.3993,0.8274,0.9258,...,compare_models,f7a788f2adb74b3683cd615d48f294a3,fbd0,56.55,Extreme Gradient Boosting,39c59009a6b5481fa6399133036d4cb3,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,d7c84cdf
4,39c59009a6b5481fa6399133036d4cb3,387310387560721782,RUNNING,mlflow-artifacts:/387310387560721782/39c59009a...,2025-04-12 17:16:10.142000+00:00,NaT,NaN,NaN,NaN,NaN,...,setup,39c59009a6b5481fa6399133036d4cb3,fbd0,0.92,Session Initialized fbd0,None,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,b62c4622
5,7c88e59b11ba450da9f890ed7034f966,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/7c88e59b1...,2025-02-25 01:19:55.033000+00:00,2025-02-25 01:19:56.659000+00:00,0.530,NaN,NaN,NaN,...,finalize_model,7c88e59b11ba450da9f890ed7034f966,8aa5,0.68,Extreme Gradient Boosting,9394b43439b543f192bd7b97e7b42a73,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,f046ceec
6,8ce14dc15acb4c9880be6790910e1581,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/8ce14dc15...,2025-02-25 01:19:53.972000+00:00,2025-02-25 01:19:54.203000+00:00,93.272,0.2154,0.7294,0.9212,...,compare_models,8ce14dc15acb4c9880be6790910e1581,8aa5,466.57,Logistic Regression,9394b43439b543f192bd7b97e7b42a73,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,dbb143d5
7,f2bae24f7f454b2eb6ac29bcc0e71238,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/f2bae24f7...,2025-02-25 01:19:53.631000+00:00,2025-02-25 01:19:53.857000+00:00,106.476,0.4702,0.8180,0.9326,...,compare_models,f2bae24f7f454b2eb6ac29bcc0e71238,8aa5,532.59,Random Forest Classifier,9394b43439b543f192bd7b97e7b42a73,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,48318b92
8,a441b19a6f1347e69e4140309fa27551,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/a441b19a6...,2025-02-25 01:19:47.272000+00:00,2025-02-25 01:19:53.515000+00:00,97.032,0.3993,0.8274,0.9258,...,compare_models,a441b19a6f1347e69e4140309fa27551,8aa5,485.36,Extreme Gradient Boosting,9394b43439b543f192bd7b97e7b42a73,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,8b414915
9,9394b43439b543f192bd7b97e7b42a73,387310387560721782,RUNNING,mlflow-artifacts:/387310387560721782/9394b4343...,2025-02-25 00:55:01.089000+00:00,NaT,NaN,NaN,NaN,NaN,...,setup,9394b43439b543f192bd7b97e7b42a73,8aa5,1.16,Session Initialized 

In [23]:
df_filtrado = df_runs[(df_runs['tags.Source'] == "finalize_model")]

In [24]:
df_filtrado

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.TT,metrics.MCC,metrics.AUC,metrics.Accuracy,...,tags.Source,tags.Run ID,tags.USI,tags.Run Time,tags.mlflow.runName,tags.mlflow.parentRunId,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.user,tags.URI
0,df71043d1c914f148d1059c28372c91a,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/df71043d1...,2025-04-12 17:20:31.790000+00:00,2025-04-12 17:20:32.854000+00:00,0.19,NaN,NaN,NaN,...,finalize_model,df71043d1c914f148d1059c28372c91a,fbd0,0.29,Extreme Gradient Boosting,39c59009a6b5481fa6399133036d4cb3,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,5cf458be
5,7c88e59b11ba450da9f890ed7034f966,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/7c88e59b1...,2025-02-25 01:19:55.033000+00:00,2025-02-25 01:19:56.659000+00:00,0.53,NaN,NaN,NaN,...,finalize_model,7c88e59b11ba450da9f890ed7034f966,8aa5,0.68,Extreme Gradient Boosting,9394b43439b543f192bd7b97e7b42a73,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,f046ceec
10,4d6478aef3d84a688d751db94641a146,387310387560721782,FINISHED,mlflow-artifacts:/387310387560721782/4d6478aef...,2025-02-18 14:17:50.047000+00:00,2025-02-18 14:17:51.025000+00:00,0.25,NaN,NaN,NaN,...,finalize_model,4d6478aef3d84a688d751db94641a146,9b8f,0.37,Extreme Gradient Boosting,0ab391445044473bb6eefae66ee422b0,LOCAL,/home/jose/.local/share/virtualenvs/data_maste...,jose,67383f49


In [25]:
run_id = df_filtrado['run_id'].values[0]

In [42]:
import pandas as pd
import mlflow
from mlflow.models.model import get_model_info
from mlflow.models import infer_signature, set_signature

In [27]:
run_id

'df71043d1c914f148d1059c28372c91a'

In [28]:

# load the logged model
model_uri = f"runs:/{run_id}/model"
model = mlflow.pyfunc.load_model(model_uri)

In [35]:
model_uri

'runs:/df71043d1c914f148d1059c28372c91a/model'

In [29]:
X_test = balenced_exp.X_test

In [43]:
mlflow.xgboost.log_model(
    artifact_path="model",
    xgb_model=model,
    # code_path=["data_master_eng_ml/models/"],
    registered_model_name="xgboost_model",
    signature=infer_signature(X_test, model.predict(X_test)),
    input_example=X_test # example input
)

AttributeError: 'PyFuncModel' object has no attribute 'save_model'

In [30]:

signature = infer_signature(X_test, model.predict(X_test))

In [31]:
# set the signature for the logged model
set_signature(model_uri, signature)


In [32]:

# now when you load the model again, it will have the desired signature
assert get_model_info(model_uri).signature == signature

In [33]:
balenced_exp.evaluate_model(best)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [34]:
# predict on test set
balenced_exp.predict_model(best)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.9319,0.8026,0.3274,0.6607,0.4379,0.4060,0.4345


,genres_first,has_remaster,age_rating_group,games_developed,has_parents,games_published,continent_name,onlinecoop,onlinecoopmax,onlinemax,...,first_person,side_view,text,third_person,unknown_player_perspectives,virtual_reality,has_global_launch,target,prediction_label,prediction_score
859,indie,False,No_Rating,3.0,0.0,1.0,Unknown,NaN,NaN,NaN,...,1,0,0,0,0,0,1,0,0,0.9575
1680,unknown_genres_name,False,No_Rating,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,1,0,0,0,0,0.9965
925,adventure,False,No_Rating,2.0,0.0,2.0,Unknown,NaN,NaN,NaN,...,0,0,0,1,0,0,1,0,0,0.9749
739,strategy,False,No_Rating,4.0,0.0,1.0,Unknown,NaN,NaN,NaN,...,1,0,0,0,0,0,1,0,0,0.8677
3640,platform,False,18+,324.0,1.0,9.0,North_America,NaN,NaN,NaN,...,0,0,0,1,0,0,1,1,1,0.9538
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2534,adventure,False,12+,2.0,0.0,2.0,Unknown,NaN,NaN,NaN,...,0,0,0,0,0,1,1,0,0,0.9573
1577,visual-novel,False,No_Rating,7.0,0.0,3.0,Asia,NaN,NaN,NaN,...,0,0,1,0,0,0,1,0,0,0.9975
3659,role-playing-rpg,False,No_Rating,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,1,0,0,0,0,0.9965
1162,fighting,False,No_Rating,2.0,0.0,1.0,Unknown,NaN,NaN,NaN,...,0,1,0,0,0,0,1,0,0,0.9095
